In [1]:
import pandas as pd
import numpy as np

In [3]:
matches = pd.read_csv('../data/wc_matches.csv')
matches['date'] = pd.to_datetime(matches['date'])
matches = matches.sort_values('date').reset_index(drop=True)

print('Total matches: ', len(matches))
print(matches.head(5))

Total matches:  1298
        date      home_team    away_team  home_score  away_score  \
0 2014-01-07          Qatar       Jordan         2.0         0.0   
1 2014-01-29         Mexico  South Korea         4.0         0.0   
2 2014-02-01  United States  South Korea         2.0         0.0   
3 2014-03-05    Switzerland      Croatia         2.0         2.0   
4 2014-03-05          Japan  New Zealand         4.0         2.0   

          tournament         city        country  neutral  is_friendly  
0  WAFF Championship         Doha          Qatar    False        False  
1           Friendly  San Antonio  United States     True         True  
2           Friendly       Carson  United States    False         True  
3           Friendly   St. Gallen    Switzerland    False         True  
4           Friendly        Tokyo          Japan    False         True  


In [7]:
#Only looking backwards at that team's last N matches
def get_team_form(team, date, matches, window=10):
    """
    Get a team's stats from their last N=10 matches
    Before a given date. Never uses future data.
    """

    past = matches[
        (matches['date'] < date) &
        (
            (matches['home_team'] == team) |
            (matches['away_team'] == team)
        )
    ].tail(window)

    #if past games are not found then these are the default value set
    if len(past) == 0:
        return{
            'goals_scored_avg': 1.2,
            'goals_conceded_avg': 1.2,
            'win_rate': 0.33,
            'draw_rate': 0.33,
            'form_points': 1.0,
            'games_played': 0
        }

    goals_scored = []
    goals_conceded = []
    results = []

    for _, row in past.iterrows():
        if row['home_team'] == team:
            scored = row['home_score']
            conceded = row['away_score']
            if scored > conceded:
                results.append(3)
            elif scored == conceded:
                results.append(1)
            else:
                results.append(0)
        else:
            scored = row['away_score']
            conceded = row['home_score']
            if scored > conceded:
                results.append(3)
            elif scored == conceded:
                results.append(1)
            else:
                results.append(0)

        goals_scored.append(scored)
        goals_conceded.append(conceded)

    return{
        'goals_scored_avg': round(np.mean(goals_scored), 3),
        'goals_conceded_avg': round(np.mean(goals_conceded), 3),
        'win_rate': round(sum(1 for r in results if r == 3) / len(results), 3),
        'draw_rate': round(sum(1  for r in results if r ==1) / len(results), 3),
        'form_points': round(np.mean(results), 3),
        'games_played': len(past)
    }


#testing it
form = get_team_form('Brazil', pd.Timestamp('2022-01-01'), matches)
print("Brazil form before 2022:", form)

Brazil form before 2022: {'goals_scored_avg': np.float64(1.4), 'goals_conceded_avg': np.float64(0.4), 'win_rate': 0.6, 'draw_rate': 0.3, 'form_points': np.float64(2.1), 'games_played': 10}


In [6]:
rankings = pd.read_csv('../data/fifa_ranking.csv')
rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])
print(rankings.columns.tolist())
print(rankings.head(5))

['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']
    rank       country_full country_abrv  total_points  previous_points  \
0  140.0  Brunei Darussalam          BRU           2.0              0.0   
1   33.0           Portugal          POR          38.0              0.0   
2   32.0             Zambia          ZAM          38.0              0.0   
3   31.0             Greece          GRE          38.0              0.0   
4   30.0            Algeria          ALG          39.0              0.0   

   rank_change confederation  rank_date  
0          140           AFC 1992-12-31  
1           33          UEFA 1992-12-31  
2           32           CAF 1992-12-31  
3           31          UEFA 1992-12-31  
4           30           CAF 1992-12-31  


In [8]:
def get_fifa_rank(team, date, rankings):
    """
    Get a team's FIFA ranking closest to a given date.
    """
    past_rankings = rankings[
        (rankings['country_full'] == team) &
        (rankings['rank_date'] <= date)
    ]

    if len(past_rankings) == 0:
        return 100 #if rank not found then default rank
    
    latest = past_rankings.sort_values('rank_date').iloc[-1]
    return int(latest['rank'])

#testing
rank = get_fifa_rank('Brazil', pd.Timestamp('2022-01-01'), rankings)
print("Brazil fifa rank before 2022:", rank)

Brazil fifa rank before 2022: 2


##### Building full training dataset

In [17]:
matches = matches.dropna(subset=['home_score', 'away_score'])

training_rows = []

for idx, match in matches.iterrows():
    date = match['date']
    home_team = match['home_team']
    away_team = match['away_team']

    # getting form for both teams
    home_form = get_team_form(home_team, date, matches, window=10)
    away_form = get_team_form(away_team, date, matches, window=10)

    #getting fifa rankings
    home_rank = get_fifa_rank(home_team, date, rankings)
    away_rank = get_fifa_rank(away_team, date, rankings)

    #calculating rank difference (positive = home team is higher ranked)
    rank_diff = away_rank - home_rank

    #target variable
    home_goals = match['home_score']
    away_goals = match['away_score']

    if home_goals > away_goals:
        outcome = 2
    elif home_goals == away_goals:
        outcome = 1
    else:
        outcome = 0

    training_rows.append({
        'date': date,
        'home_team': home_team,
        'away_team': away_team,

        'home_goals_scored_avg': home_form['goals_scored_avg'],
        'home_goals_conceded_avg': home_form['goals_conceded_avg'],
        'home_win_rate': home_form['win_rate'],
        'home_draw_rate': home_form['draw_rate'],
        'home_form_points': home_form['form_points'],
        'home_games_played': home_form['games_played'],
        'home_rank': home_rank,

        'away_goals_scored_avg': away_form['goals_scored_avg'],
        'away_goals_conceded_avg': away_form['goals_conceded_avg'],
        'away_win_rate': away_form['win_rate'],
        'away_draw_rate': away_form['draw_rate'],
        'away_form_points': away_form['form_points'],
        'away_games_played': away_form['games_played'],
        'away_rank': away_rank,

        'rank_diff': rank_diff,
        'goal_avg_diff': home_form['goals_scored_avg'] - away_form['goals_scored_avg'],
        'form_diff': home_form['form_points'] - away_form['form_points'],

        'home_goals': int(home_goals),
        'away_goals': int(away_goals),
        'outcome': outcome
    
    })
    
    if idx % 500  == 0:
        print(f" Processed {idx}/{len(matches)} matches...")

training_df = pd.DataFrame(training_rows)
print("Training Data Shape: ", training_df.shape)
print(training_df.head(5))

 Processed 0/1241 matches...
 Processed 500/1241 matches...
 Processed 1000/1241 matches...
Training Data Shape:  (1241, 23)
        date      home_team    away_team  home_goals_scored_avg  \
0 2014-01-07          Qatar       Jordan                    1.2   
1 2014-01-29         Mexico  South Korea                    1.2   
2 2014-02-01  United States  South Korea                    1.2   
3 2014-03-05    Switzerland      Croatia                    1.2   
4 2014-03-05          Japan  New Zealand                    1.2   

   home_goals_conceded_avg  home_win_rate  home_draw_rate  home_form_points  \
0                      1.2           0.33            0.33               1.0   
1                      1.2           0.33            0.33               1.0   
2                      1.2           0.33            0.33               1.0   
3                      1.2           0.33            0.33               1.0   
4                      1.2           0.33            0.33               1.0  

In [18]:
print("Outcome distribution: ")
outcome_map = {2: 'Home Win', 1: 'Draw', 0: 'Away Win'}
print(training_df['outcome'].map(outcome_map).value_counts())

print("\nFeature Summary: ")
print(training_df.describe().round(2))

print("\nMissing values: ")
print(training_df.isnull().sum())

Outcome distribution: 
outcome
Home Win    563
Away Win    371
Draw        307
Name: count, dtype: int64

Feature Summary: 
                             date  home_goals_scored_avg  \
count                        1241                1241.00   
mean   2020-04-17 04:22:14.407735                   1.34   
min           2014-01-07 00:00:00                   0.00   
25%           2017-06-01 00:00:00                   1.00   
50%           2020-09-08 00:00:00                   1.30   
75%           2023-06-14 00:00:00                   1.70   
max           2026-03-31 00:00:00                   5.00   
std                           NaN                   0.56   

       home_goals_conceded_avg  home_win_rate  home_draw_rate  \
count                  1241.00        1241.00         1241.00   
mean                      1.26           0.39            0.24   
min                       0.00           0.00            0.00   
25%                       0.90           0.22            0.10   
50%       

#### Saving the dataset

In [20]:
training_df.to_csv('../data/training_data.csv', index=False)
print(training_df.shape)

(1241, 23)
